# Elise DV Ablation

Runtime ablation for the 10D velocity-delta inputs (`dv_x`, `dv_y`) on Micro-Elise #2 and Elise-264-GSTP.

In [ ]:
# cell: setup
from pathlib import Path
import sys

import pandas as pd


def repo_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / "dqn/src").exists() and (path / "hpo/src").exists() and (path / "nn_viz/src").exists():
            return path
    raise RuntimeError(f"Could not find repo root from {start}")


REPO_ROOT = repo_root()
for source_root in ["dqn/src", "hpo/src", "distillation/src", "nn_viz/src"]:
    sys.path.insert(0, str(REPO_ROOT / source_root))

from distillation.dataset import _load_teacher
from hpo.checkpointing import checkpoint_metadata
from hpo.environments.solar_system_lander.env import DEFAULT_WORLD_MIX, EnvFactory
from nn_viz import DEFAULT_INPUT_ABLATIONS, evaluate_input_ablations, load_student_network

In [ ]:
# cell: load-models; requires: setup
RESULT_DIR = REPO_ROOT / "nn_viz" / "_local_runs" / "micro_elise_02"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

MICRO_CHECKPOINT_PATH = Path(r"G:\Meine Ablage\rl_lab\distillation\runs\solar_system_lander_10d_elise_stp_student_64x64_20260721T161456Z\student_checkpoint.pt")
TEACHER_CHECKPOINT_PATH = Path(r"G:\Meine Ablage\rl_lab\hpo\best_checkpoints\solar_system_lander_10d_elise_stp\best_eval_checkpoint.pt")
ABLATION_SEEDS = range(100)

env_factory = EnvFactory("10d", world_mix=DEFAULT_WORLD_MIX)
micro_q_net = load_student_network(MICRO_CHECKPOINT_PATH)
teacher_metadata = checkpoint_metadata(TEACHER_CHECKPOINT_PATH)
teacher_q_net = _load_teacher(
    TEACHER_CHECKPOINT_PATH,
    env_factory,
    next(iter(DEFAULT_WORLD_MIX)),
    device="cpu",
    metadata=teacher_metadata,
)

print(f"results: {RESULT_DIR}")

In [ ]:
# cell: helpers; requires: setup
def ordered_ablation_summary(rows):
    ablation_order = {spec.name: index for index, spec in enumerate(DEFAULT_INPUT_ABLATIONS)}
    return (
        pd.DataFrame(rows)
        .assign(ablation_order=lambda df: df.ablation.map(ablation_order))
        .sort_values(["world", "ablation_order"])
        .drop(columns="ablation_order")
        .reset_index(drop=True)
    )


def aggregate_ablation_summary(summary, model):
    rows = summary.query('ablation != "normal"').copy()
    return (
        rows.groupby("ablation", sort=False)
        .agg(
            worse_worlds=("delta_vs_normal", lambda values: int((values < 0).sum())),
            mean_delta=("delta_vs_normal", "mean"),
            mean_action_agreement=("action_agreement", "mean"),
            mean_q_delta=("q_delta", "mean"),
        )
        .assign(model=model)
        .reset_index()
        [["model", "ablation", "worse_worlds", "mean_delta", "mean_action_agreement", "mean_q_delta"]]
    )

In [ ]:
# cell: micro-elise-dv-ablation; requires: load-models, helpers
micro_ablation_summary = ordered_ablation_summary(
    evaluate_input_ablations(micro_q_net, env_factory, DEFAULT_WORLD_MIX.keys(), ABLATION_SEEDS)
)
micro_ablation_summary.to_csv(RESULT_DIR / "input_ablation_summary.csv", index=False)

print(f"saved Micro-Elise DV ablation to {RESULT_DIR}")
display(micro_ablation_summary.round({"mean_score": 1, "delta_vs_normal": 1, "mean_steps": 1, "action_agreement": 3, "q_delta": 3}))

In [ ]:
# cell: teacher-dv-ablation; requires: load-models, helpers
teacher_ablation_summary = ordered_ablation_summary(
    evaluate_input_ablations(teacher_q_net, env_factory, DEFAULT_WORLD_MIX.keys(), ABLATION_SEEDS)
)
teacher_ablation_summary.to_csv(RESULT_DIR / "teacher_input_ablation_summary.csv", index=False)

print(f"saved Elise-264 DV ablation to {RESULT_DIR}")
display(teacher_ablation_summary.round({"mean_score": 1, "delta_vs_normal": 1, "mean_steps": 1, "action_agreement": 3, "q_delta": 3}))

In [ ]:
# cell: compare-results; requires: micro-elise-dv-ablation, teacher-dv-ablation
comparison = pd.concat(
    [
        aggregate_ablation_summary(micro_ablation_summary, "Micro-Elise #2"),
        aggregate_ablation_summary(teacher_ablation_summary, "Elise-264-GSTP"),
    ],
    ignore_index=True,
)
comparison.to_csv(RESULT_DIR / "dv_ablation_comparison.csv", index=False)
display(comparison.round({"mean_delta": 1, "mean_action_agreement": 3, "mean_q_delta": 3}))